In [154]:
from supabase import create_client
from dotenv import load_dotenv, find_dotenv
import os
import pandas as pd

env_path = find_dotenv()
load_dotenv(env_path, override=True)

supabase_url = os.getenv("SUPABASE_URL")
supabase_key = os.getenv("SUPABASE_ANON_KEY")

print("URL olemas:", supabase_url is not None)
print("KEY olemas:", supabase_key is not None)

supabase = create_client(
    supabase_url,
    supabase_key
)

df_orders = pd.DataFrame(
    supabase.table('sales').select('*').execute().data
)

df_customers = pd.DataFrame(
    supabase.table('customers').select('*').execute().data
)

print(f"Tellimusi: {len(df_orders)}, Kliente: {len(df_customers)}")

#Mitu rida ja veergu on

print("Sales tabeli read ja veerud:", df_orders.shape)
print("Customers tabeli read ja veerud:", df_customers.shape)

#Millised on veerud tabelites sales ja customers
print("Sales tabeli read ja veerud:", df_orders.dtypes)
print("Customers tabeli read ja veerud:", df_customers.dtypes)

URL olemas: True
KEY olemas: True


2026-06-22 13:28:42,168 - INFO - HTTP Request: GET https://llzinozmlmlispovzjww.supabase.co/rest/v1/sales?select=%2A "HTTP/2 200 OK"
2026-06-22 13:28:42,284 - INFO - HTTP Request: GET https://llzinozmlmlispovzjww.supabase.co/rest/v1/customers?select=%2A "HTTP/2 200 OK"


Tellimusi: 1000, Kliente: 1000
Sales tabeli read ja veerud: (1000, 12)
Customers tabeli read ja veerud: (1000, 9)
Sales tabeli read ja veerud: id                  int64
sale_id             int64
invoice_id            str
sale_date             str
customer_id       float64
product_id          int64
quantity            int64
unit_price        float64
total_price       float64
channel               str
store_location        str
payment_method        str
dtype: object
Customers tabeli read ja veerud: customer_id          int64
first_name             str
last_name              str
email                  str
phone                  str
city                   str
registration_date      str
loyalty_tier           str
birth_year           int64
dtype: object


In [155]:
import os
from pathlib import Path

print("Notebooki töökaust:")
print(Path.cwd())

print("\nSelles kaustas olevad failid:")
print(os.listdir())

Notebooki töökaust:
c:\Users\kasutaja\Documents\DACA-Python\week-8

Selles kaustas olevad failid:
['monthly_chart_20260622_1141.html', 'monthly_city_chart_20260622_1307.html', 'monthly_city_chart_20260622_1316.html', 'monthly_city_chart_20260622_1319.html', 'monthly_city_chart_20260622_1324.html', 'monthly_city_report_20260622_1307.csv', 'monthly_city_report_20260622_1316.csv', 'monthly_city_report_20260622_1319.csv', 'monthly_city_report_20260622_1324.csv', 'monthly_report_20260622_1141.csv', 'monthly_report_20260622_1215.csv', 'monthly_report_20260622_1307.csv', 'monthly_report_20260622_1316.csv', 'monthly_report_20260622_1319.csv', 'monthly_report_20260622_1324.csv', 'rfm_segments_20260622_1319.csv', 'rfm_segments_20260622_1324.csv', 'week_8_iseseisevtöö.ipynb']


Konkreetse linna andmed Tallinn

In [156]:
import pandas as pd

response = (
    supabase
    .table('sales')
    .select('*')
    .eq('store_location', 'Tallinn')
    .order('total_price', desc=True)
    .execute()
)

df_tallinn = pd.DataFrame(response.data)

print(df_tallinn.head())

print(
    f"Tallinna tellimusi: {len(df_tallinn)}, "
    f"Käive: {df_tallinn['total_price'].sum():.2f} EUR"
)


2026-06-22 13:28:42,445 - INFO - HTTP Request: GET https://llzinozmlmlispovzjww.supabase.co/rest/v1/sales?select=%2A&store_location=eq.Tallinn&order=total_price.desc "HTTP/2 200 OK"


     id  sale_id        invoice_id            sale_date  customer_id  \
0  8379     8379  INV-202410-00273  2024-10-06T00:00:00       2712.0   
1  3441     3441  INV-202310-00264  2023-10-03T00:00:00       2707.0   
2  7543     7543  INV-202408-00341  2024-08-06T00:00:00       2961.0   
3  9684     9684  INV-202501-00242  2025-01-15T00:00:00       2288.0   
4   144      144  INV-202301-00144  2023-01-07T00:00:00       3446.0   

   product_id  quantity  unit_price  total_price channel store_location  \
0        1210         5      374.54      1872.70    pood        Tallinn   
1        1134         5      371.79      1858.95    pood        Tallinn   
2        1042         5      347.84      1739.20    pood        Tallinn   
3        1031         4      434.08      1736.32    pood        Tallinn   
4        1009         5      332.55      1662.75    pood        Tallinn   

  payment_method  
0          kaart  
1       sularaha  
2          kaart  
3          kaart  
4      järelmaks  
Ta

Andmed Tartu kohta

In [157]:
import pandas as pd

response = (
    supabase
    .table('sales')
    .select('*')
    .eq('store_location', 'Tartu')
    .execute()
)

df_tartu = pd.DataFrame(response.data)

tellimuste_arv = len(df_tartu)
kogukaive = df_tartu['total_price'].sum()
keskmine_tellimus = df_tartu['total_price'].mean()

print("Tartu tulemused")
print("Tellimuste arv:", tellimuste_arv)
print(f"Kogukäive: {kogukaive:.2f} EUR")
print(f"Keskmine tellimus: {keskmine_tellimus:.2f} EUR")

2026-06-22 13:28:42,608 - INFO - HTTP Request: GET https://llzinozmlmlispovzjww.supabase.co/rest/v1/sales?select=%2A&store_location=eq.Tartu "HTTP/2 200 OK"


Tartu tulemused
Tellimuste arv: 1000
Kogukäive: 285232.95 EUR
Keskmine tellimus: 285.23 EUR


Viimase 30 päeva tellimused

In [158]:
#viitekuupäev
today = pd.to_datetime("2025-02-28")

start_date = today - pd.Timedelta(days=30)

Supabase API päring viimase 30 päeva kohta

In [159]:
import pandas as pd

today = pd.to_datetime("2025-02-28")
start_date = today - pd.Timedelta(days=30)

response = (
    supabase
    .table("sales")
    .select("*")
    .gte("sale_date", start_date.strftime("%Y-%m-%d"))
    .lte("sale_date", today.strftime("%Y-%m-%d"))
    .execute()
)

df_last_30_days = pd.DataFrame(response.data)

print("Viimase 30 päeva tehinguid:", len(df_last_30_days))
print("Kuupäevavahemik:", start_date.strftime("%Y-%m-%d"), "kuni", today.strftime("%Y-%m-%d"))
print(df_last_30_days.head())

2026-06-22 13:28:42,751 - INFO - HTTP Request: GET https://llzinozmlmlispovzjww.supabase.co/rest/v1/sales?select=%2A&sale_date=gte.2025-01-29&sale_date=lte.2025-02-28 "HTTP/2 200 OK"


Viimase 30 päeva tehinguid: 380
Kuupäevavahemik: 2025-01-29 kuni 2025-02-28
     id  sale_id        invoice_id            sale_date  customer_id  \
0  9476     9476  INV-202501-00034  2025-01-29T00:00:00       2639.0   
1  9482     9482  INV-202501-00040  2025-01-29T00:00:00       3928.0   
2  9513     9513  INV-202501-00071  2025-01-29T00:00:00       3301.0   
3  9530     9530  INV-202501-00088  2025-01-29T00:00:00       2244.0   
4  9597     9597  INV-202501-00155  2025-01-29T00:00:00       2565.0   

   product_id  quantity  unit_price  total_price channel store_location  \
0        1064         1       95.87        95.87    pood        Tallinn   
1        1299         2       83.10       166.20    pood        Tallinn   
2        1110         1      151.08       151.08    pood        Tallinn   
3        1308         2       67.75       135.50  online            NaN   
4        1203         2       17.97        35.94  online            NaN   

  payment_method  
0      järelmaks  
1 

In [160]:
def city_report(df, city):
    """Genereeri asukohapõhine müügiraport."""
    city_data = df[df['store_location'] == city]

    return {
        'store_location': city,
        'orders': len(city_data),
        'revenue': city_data['total_price'].sum()
    }


for city in ['Tallinn', 'Tartu', 'Pärnu']:
    r = city_report(df_orders, city)

    print(
        f"{r['store_location']}: "
        f"{r['orders']} tellimust, "
        f"{r['revenue']:.2f} EUR"
    )

Tallinn: 398 tellimust, 112060.36 EUR
Tartu: 175 tellimust, 51383.15 EUR
Pärnu: 115 tellimust, 30989.46 EUR


Logimine ja vigade käivitamine

In [161]:
import logging

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

def safe_fetch(supabase_client, table_name):
    """Too andmed turvalise vigade käsitlemisega."""
    try:
        response = supabase_client.table(table_name).select('*').execute()
        df = pd.DataFrame(response.data)
        if len(df) == 0:
            logger.warning(f"Tabel '{table_name}' on tühi!")
        logger.info(f"Laaditud {len(df)} rida tabelist '{table_name}'")
        return df
    except Exception as e:
        logger.error(f"Viga tabeli '{table_name}' lugemisel: {e}")
        return pd.DataFrame()  # Tagasta tühi DataFrame vea korral


2.3 Concrete Practice: Automatiseerimise harjutused

In [162]:
from datetime import datetime

def weekly_sales_report(df, report_date=None):
    """Genereeri iganädalane müügiraport.

    Args:
        df: DataFrame müügitellimustega
        report_date: Raporti kuupäev (vaikimisi täna)
    Returns:
        dict: Raporti kokkuvõte
    """
    if report_date is None:
        report_date = datetime.now().strftime('%Y-%m-%d')
    return {
        'report_date': report_date,
        'total_orders': len(df),
        'total_revenue': round(df['total_price'].sum(), 2),
        'avg_order': round(df['total_price'].mean(), 2),
    }

# Käivita
result = weekly_sales_report(df_orders)
for key, value in result.items():
    print(f"  {key}: {value}")


  report_date: 2026-06-22
  total_orders: 1000
  total_revenue: 285483.53
  avg_order: 285.48


Ülesanne: Kirjuta funktsioon, mis automatiseerib eelmise nädala RFM analüüsi. Funktsiooni sisend on DataFrame ja viitekuupäev, väljund on RFM DataFrame segmentidega.


In [163]:
def calculate_rfm(df, reference_date=None):
    """Arvuta RFM skoorid ja segmendid.

    Args:
        df: DataFrame tellimustega (veerud: customer_id, sale_date, total_price)
        reference_date: Viitekuupäev Recency arvutamiseks

    Returns:
        DataFrame: RFM skoorid ja segmendid iga kliendi kohta
    """
    if reference_date is None:
        reference_date = pd.to_datetime('today')
    else:
        reference_date = pd.to_datetime(reference_date)

    df['sale_date'] = pd.to_datetime(df['sale_date'])

    # Recency: päevi viimasest ostust
    recency = df.groupby('customer_id')['sale_date'].max().reset_index()
    recency.columns = ['customer_id', 'last_purchase']
    recency['recency_days'] = (reference_date - recency['last_purchase']).dt.days

    # Frequency: ostude arv
    frequency = df.groupby('customer_id').size().reset_index(name='frequency')

    # Monetary: kogukulutus
    monetary = df.groupby('customer_id')['total_price'].sum().reset_index()  # Täida!
    monetary.columns = ['customer_id', 'monetary']

    # Liida kokku
    rfm = recency[['customer_id', 'recency_days']].merge(
        frequency, on='customer_id'
    ).merge(
        monetary, on='customer_id'
    )

    # Skooride määramine (lihtsustatud)
    rfm['R_score'] = pd.qcut(rfm['recency_days'], q=3, labels=[3, 2, 1]).astype(int)
    rfm['F_score'] = pd.qcut(
        rfm['frequency'].rank(method='first'), q=3, labels=[1, 2, 3]
    ).astype(int)
    rfm['M_score'] = pd.qcut(rfm['monetary'], q=3, labels=[1, 2, 3]).astype(int)
    rfm['RFM_score'] = rfm['R_score'] + rfm['F_score'] + rfm['M_score']

    # Segmenteerimine
    def assign_segment(score):
        if score >= 8:
            return 'VIP Champion'              # Täida! Vihje: 'VIP Champions'
        elif score >= 6:
            return 'Loyal Customers'              # Täida!
        elif score >= 4:
            return 'Regular Customers'              # Täida!
        else:
            return 'New Customers'              # Täida!

    rfm['segment'] = rfm['RFM_score'].apply(assign_segment)
    return rfm

# Testi
rfm_result = calculate_rfm(df_orders, reference_date='2024-08-01')
print(rfm_result.sort_values('RFM_score', ascending=False))
print(f"\nSegmentide jaotus:")
print(rfm_result['segment'].value_counts())

     customer_id  recency_days  frequency  monetary  R_score  F_score  \
6         2029.0           497          2   1796.84        3        3   
11        2054.0           474          2    512.73        3        3   
375       3709.0          -488          2    581.63        3        3   
376       3710.0           491          2    419.74        3        3   
371       3696.0           463          3   1910.88        3        3   
..           ...           ...        ...       ...      ...      ...   
278       3308.0           572          1     22.04        1        1   
50        2247.0           560          1     95.87        1        1   
92        2439.0           560          1    107.67        1        1   
56        2261.0           542          1     73.18        1        1   
35        2202.0           553          1   -308.84        1        1   

     M_score  RFM_score        segment  
6          3          9   VIP Champion  
11         3          9   VIP Champion  


5 TOP toodet iganädalaselt müügikohtade järgi

In [164]:
print(df_orders.columns.tolist())

['id', 'sale_id', 'invoice_id', 'sale_date', 'customer_id', 'product_id', 'quantity', 'unit_price', 'total_price', 'channel', 'store_location', 'payment_method']


In [165]:
import pandas as pd

def weekly_top_products(
    df,
    reference_date='2024-12-31',
    locations=['Tallinn', 'Tartu', 'Pärnu', 'online']
):
    """
    Iganädalane raport:
    TOP 5 toodet product_id järgi Tallinnas, Tartus, Pärnus ja online kanalis.
    """

    df = df.copy()

    # Kuupäevad õigeks
    df['sale_date'] = pd.to_datetime(
        df['sale_date'],
        dayfirst=True,
        errors='coerce'
    )

    # Hind numbriks
    df['total_price'] = pd.to_numeric(
        df['total_price'],
        errors='coerce'
    )

    # Viitekuupäev ja nädala algus
    reference_date = pd.to_datetime(reference_date)
    start_date = reference_date - pd.Timedelta(days=6)

    # Puhasta vigased read
    df = df.dropna(
        subset=['sale_date', 'product_id', 'total_price']
    )

    # Tallinn, Tartu, Pärnu tulevad store_location veerust
    city_data = df[
        (df['sale_date'] >= start_date) &
        (df['sale_date'] <= reference_date) &
        (df['store_location'].isin(['Tallinn', 'Tartu', 'Pärnu']))
    ].copy()

    city_data['report_location'] = city_data['store_location']

    # Online tuleb channel veerust
    online_data = df[
        (df['sale_date'] >= start_date) &
        (df['sale_date'] <= reference_date) &
        (df['channel'].str.lower() == 'online')
    ].copy()

    online_data['report_location'] = 'online'

    # Pane linnad ja online kokku
    report_data = pd.concat(
        [city_data, online_data],
        ignore_index=True
    )

    # Grupeeri asukoha ja product_id järgi
    summary = (
        report_data
        .groupby(['report_location', 'product_id'])
        .agg(
            tellimuste_arv=('sale_id', 'count'),
            kogus=('quantity', 'sum'),
            kogukaive=('total_price', 'sum'),
            keskmine_tellimus=('total_price', 'mean')
        )
        .reset_index()
    )

    # Sorteeri iga asukoha sees käibe järgi
    summary = summary.sort_values(
        ['report_location', 'kogukaive'],
        ascending=[True, False]
    )

    # Lisa koht
    summary['koht'] = (
        summary
        .groupby('report_location')['kogukaive']
        .rank(method='first', ascending=False)
        .astype(int)
    )

    # Võta TOP 5 iga asukoha kohta
    top5 = (
        summary[summary['koht'] <= 5]
        .sort_values(['report_location', 'koht'])
        .reset_index(drop=True)
    )

    print(f"Periood: {start_date.date()} kuni {reference_date.date()}")

    return top5

In [166]:
top5_weekly = weekly_top_products(
    df_orders,
    reference_date='2024-12-31'
)

print(top5_weekly)

for location in ['Tallinn', 'Tartu', 'Pärnu', 'online']:
    print("\n" + location)
    print("-" * 40)

    display(
        top5_weekly[
            top5_weekly['report_location'] == location
        ][
            [
                'koht',
                'product_id',
                'tellimuste_arv',
                'kogus',
                'kogukaive',
                'keskmine_tellimus'
            ]
        ]
    )

Periood: 2024-12-25 kuni 2024-12-31
Empty DataFrame
Columns: [report_location, product_id, tellimuste_arv, kogus, kogukaive, keskmine_tellimus, koht]
Index: []

Tallinn
----------------------------------------


,koht,product_id,tellimuste_arv,kogus,kogukaive,keskmine_tellimus



Tartu
----------------------------------------


,koht,product_id,tellimuste_arv,kogus,kogukaive,keskmine_tellimus



Pärnu
----------------------------------------


,koht,product_id,tellimuste_arv,kogus,kogukaive,keskmine_tellimus



online
----------------------------------------


,koht,product_id,tellimuste_arv,kogus,kogukaive,keskmine_tellimus


In [167]:
df_check = df_orders.copy()

df_check['sale_date'] = pd.to_datetime(
    df_check['sale_date'],
    errors='coerce'
)

print("Min kuupäev:", df_check['sale_date'].min())
print("Max kuupäev:", df_check['sale_date'].max())
print("Puuduvaid kuupäevi:", df_check['sale_date'].isna().sum())

Min kuupäev: 2023-01-01 00:00:00
Max kuupäev: 2026-03-17 00:00:00
Puuduvaid kuupäevi: 0


In [168]:
import pandas as pd

df_check = df_orders.copy()

df_check['sale_date'] = pd.to_datetime(
    df_check['sale_date'],
    errors='coerce'
)

reference_date = pd.to_datetime('2024-12-31')
start_date = reference_date - pd.Timedelta(days=6)
end_date = reference_date + pd.Timedelta(days=1)

weekly_data = df_check[
    (df_check['sale_date'] >= start_date) &
    (df_check['sale_date'] < end_date)
].copy()

print("Periood:", start_date.date(), "kuni", reference_date.date())
print("Ridu selles perioodis:", len(weekly_data))

print("\nStore location väärtused selles perioodis:")
print(weekly_data['store_location'].value_counts(dropna=False))

print("\nChannel väärtused selles perioodis:")
print(weekly_data['channel'].value_counts(dropna=False))

print("\nNäidisread:")
print(
    weekly_data[
        ['sale_date', 'store_location', 'channel', 'product_id', 'quantity', 'total_price']
    ].head(20)
)

Periood: 2024-12-25 kuni 2024-12-31
Ridu selles perioodis: 0

Store location väärtused selles perioodis:
Series([], Name: count, dtype: int64)

Channel väärtused selles perioodis:
Series([], Name: count, dtype: int64)

Näidisread:
Empty DataFrame
Columns: [sale_date, store_location, channel, product_id, quantity, total_price]
Index: []


In [169]:
df_check = df_orders.copy()

df_check['sale_date'] = pd.to_datetime(
    df_check['sale_date'],
    errors='coerce'
)

before_date = pd.to_datetime('2024-12-31')

last_available_before = df_check.loc[
    df_check['sale_date'] <= before_date,
    'sale_date'
].max()

print("Viimane andmetega kuupäev enne 2024-12-31:", last_available_before)

Viimane andmetega kuupäev enne 2024-12-31: 2023-04-30 00:00:00


In [170]:
top5_weekly = weekly_top_products(
    df_orders,
    reference_date=last_available_before
)

print(top5_weekly)

Periood: 2023-04-24 kuni 2023-04-30
   report_location  product_id  tellimuste_arv  kogus  kogukaive  \
0            Pärnu        1136               1      5     471.25   
1          Tallinn        1143               2      2     615.66   
2          Tallinn        1265               1      2     539.70   
3          Tallinn        1083               1      5     360.70   
4          Tallinn        1330               1      2     334.46   
5          Tallinn        1006               1      1     319.42   
6            Tartu        1111               1      5    1635.35   
7            Tartu        1227               1      3     360.51   
8            Tartu        1234               1      2     325.72   
9            Tartu        1223               1      1     319.98   
10           Tartu        1201               1      2     294.32   
11          online        1042               1      3    1043.52   
12          online        1167               1      2     541.98   
13          

Samm 2: Pipeline struktuur koodis

In [171]:
# === EXTRACT ===
def extract_data(supabase_client):
    """Too andmed Supabase'ist."""
    orders = pd.DataFrame(supabase_client.table('sales').select('*').execute().data)
    customers = pd.DataFrame(supabase_client.table('customers').select('*').execute().data)
    return orders, customers

# === TRANSFORM ===
def transform_data(orders, customers):
    """Puhasta ja analüüsi andmeid."""
    df = pd.merge(orders, customers, on='customer_id', how='left')
    df['sale_date'] = pd.to_datetime(df['sale_date'])
    rfm = calculate_rfm(df)  # Osa 2 funktsioon!
    return df, rfm

# === LOAD ===
def load_results(rfm, output_dir='reports'):
    """Salvesta tulemused."""
    timestamp = datetime.now().strftime('%Y%m%d_%H%M')
    rfm.to_csv(f'{output_dir}/rfm_{timestamp}.csv', index=False)
    px.bar(rfm['segment'].value_counts().reset_index(), x='segment', y='count',
           title=f'RFM Segmendid ({timestamp})').write_html(f'{output_dir}/rfm_{timestamp}.html')

Samm 3: Pipeline orkestreerimie ja valideerimine

In [172]:
def validate(df):
    """Kontrolli andmete kvaliteeti."""
    if len(df) == 0:
        logger.warning("VALIDATE: Tühi andmestik!")
        return False
    return True

def run_pipeline():
    """Käivita kogu RFM pipeline."""
    try:
        orders, customers = extract_data(supabase)
        if not validate(orders):
            return False
        df, rfm = transform_data(orders, customers)
        load_results(rfm)
        return True
    except Exception as e:
        logger.error(f"PIPELINE FAILED: {e}")
        return False


Ülesanne: Kopeeri ja käivita järgmine lihtsustatud pipeline, mis töötab ka ilma Supabase ühenduseta.

In [173]:
import pandas as pd
import plotly.express as px
from datetime import datetime

# === EXTRACT ===
def extract_orders():
    """Simuleeri andmete toomist API-st."""
    print("[EXTRACT] Laadin...")

    data = {
        'customer_id': [1001,1002,1003,1001,1002,1004,1003,1001,1005,1004,
                        1002,1003,1005,1001,1006,1004,1002,1007,1003,1005],
        'sale_date': pd.date_range('2024-01-15', periods=20, freq='10D'),
        'total_price': [89.99,45.50,120.00,67.30,55.00,210.00,33.50,145.00,
                        78.00,92.00,160.00,44.00,88.50,230.00,37.00,175.00,
                        110.00,65.00,95.00,125.00],
        'store_location': ['Tallinn','Tartu','Tallinn','Tallinn','Tartu','Parnu','Tallinn',
                           'Tallinn','Tartu','Parnu','Tartu','Tallinn','Tartu','Tallinn',
                           'Parnu','Parnu','Tartu','Tallinn','Tallinn','Tartu']
    }

    df = pd.DataFrame(data)
    print(f"[EXTRACT] {len(df)} tellimust laaditud")

    return df


# === TRANSFORM ===
def transform_monthly(df):
    """Arvuta kuuraport."""
    print("[TRANSFORM] Arvutan...")

    df = df.copy()
    df['sale_date'] = pd.to_datetime(df['sale_date'])

    monthly = (
        df.groupby(df['sale_date'].dt.to_period('M'))
        .agg(
            tellimusi=('sale_date', 'count'),
            kaive=('total_price', 'sum')
        )
        .reset_index()
    )

    monthly['sale_date'] = monthly['sale_date'].astype(str)
    monthly['kaive'] = monthly['kaive'].round(2)

    print(f"[TRANSFORM] {len(monthly)} kuud")

    return monthly


# === LOAD ===
def load_report(monthly):
    """Salvesta CSV ja kuva graafik."""
    print("[LOAD] Alustan...")

    ts = datetime.now().strftime('%Y%m%d_%H%M')

    monthly.to_csv(
        f'monthly_report_{ts}.csv',
        index=False,
        encoding='utf-8-sig'
    )

    fig = px.bar(
        monthly,
        x='sale_date',
        y='kaive',
        title=f'UrbanStyle kuukäive ({ts})',
        labels={
            'sale_date': 'Kuu',
            'kaive': 'Käive (EUR)'
        }
    )

    fig.show()

    print("[LOAD] CSV salvestatud ja graafik kuvatud")


# === RUN ===
print("PIPELINE START")

df = extract_orders()
monthly = transform_monthly(df)
load_report(monthly)

print("PIPELINE COMPLETE")
print(monthly.to_string(index=False))

PIPELINE START
[EXTRACT] Laadin...
[EXTRACT] 20 tellimust laaditud
[TRANSFORM] Arvutan...
[TRANSFORM] 7 kuud
[LOAD] Alustan...


[LOAD] CSV salvestatud ja graafik kuvatud
PIPELINE COMPLETE
sale_date  tellimusi  kaive
  2024-01          2 135.49
  2024-02          3 242.30
  2024-03          3 388.50
  2024-04          3 330.00
  2024-05          3 362.50
  2024-06          3 322.00
  2024-07          3 285.00


In [174]:
def load_report(monthly):
    """Salvesta ainult CSV ja kuva graafik."""
    ts = datetime.now().strftime('%Y%m%d_%H%M')

    print("[LOAD] Salvestan CSV...")
    monthly.to_csv(
        f'monthly_report_{ts}.csv',
        index=False,
        encoding='utf-8-sig'
    )

    print("[LOAD] Loon graafiku...")
    fig = px.bar(
        monthly,
        x='sale_date',
        y='kaive',
        title=f'UrbanStyle kuukäive ({ts})',
        labels={
            'sale_date': 'Kuu',
            'kaive': 'Käive (EUR)'
        }
    )

    fig.show()

    print("[LOAD] CSV salvestatud ja graafik kuvatud")

    print("PIPELINE START")
df = extract_orders()
monthly = transform_monthly(df)
load_report(monthly)
print("PIPELINE COMPLETE")
print(monthly.to_string(index=False))

[EXTRACT] Laadin...
[EXTRACT] 20 tellimust laaditud
[TRANSFORM] Arvutan...
[TRANSFORM] 7 kuud
[LOAD] Salvestan CSV...
[LOAD] Loon graafiku...


[LOAD] CSV salvestatud ja graafik kuvatud
PIPELINE START
PIPELINE COMPLETE
sale_date  tellimusi  kaive
  2024-01          2 135.49
  2024-02          3 242.30
  2024-03          3 388.50
  2024-04          3 330.00
  2024-05          3 362.50
  2024-06          3 322.00
  2024-07          3 285.00


Linnade kaupa

In [175]:
import pandas as pd
import plotly.express as px
from datetime import datetime

# === EXTRACT ===
def extract_orders():
    """Simuleeri andmete toomist API-st."""
    print("[EXTRACT] Laadin...")

    data = {
        'customer_id': [1001,1002,1003,1001,1002,1004,1003,1001,1005,1004,
                        1002,1003,1005,1001,1006,1004,1002,1007,1003,1005],
        'sale_date': pd.date_range('2024-01-15', periods=20, freq='10D'),
        'total_price': [89.99,45.50,120.00,67.30,55.00,210.00,33.50,145.00,
                        78.00,92.00,160.00,44.00,88.50,230.00,37.00,175.00,
                        110.00,65.00,95.00,125.00],
        'store_location': ['Tallinn','Tartu','Tallinn','Tallinn','Tartu','Parnu','Tallinn',
                           'Tallinn','Tartu','Parnu','Tartu','Tallinn','Tartu','Tallinn',
                           'Parnu','Parnu','Tartu','Tallinn','Tallinn','Tartu']
    }

    df = pd.DataFrame(data)

    print(f"[EXTRACT] {len(df)} tellimust laaditud")

    return df


# === TRANSFORM ===
def transform_monthly_by_city(df):
    """Arvuta kuuraport linnade kaupa."""
    print("[TRANSFORM] Arvutan kuuraporti linnade kaupa...")

    df = df.copy()

    df['sale_date'] = pd.to_datetime(df['sale_date'])

    df['month'] = df['sale_date'].dt.to_period('M').astype(str)

    monthly_city = (
        df.groupby(['month', 'store_location'])
        .agg(
            tellimusi=('sale_date', 'count'),
            kaive=('total_price', 'sum'),
            keskmine_tellimus=('total_price', 'mean')
        )
        .reset_index()
    )

    monthly_city['kaive'] = monthly_city['kaive'].round(2)
    monthly_city['keskmine_tellimus'] = monthly_city['keskmine_tellimus'].round(2)

    print(f"[TRANSFORM] {len(monthly_city)} kuu-linna rida")

    return monthly_city


# === LOAD ===
def load_report(monthly_city):
    """Salvesta CSV ja graafik."""
    ts = datetime.now().strftime('%Y%m%d_%H%M')

    monthly_city.to_csv(
        f'monthly_city_report_{ts}.csv',
        index=False,
        encoding='utf-8-sig'
    )

    fig = px.bar(
        monthly_city,
        x='month',
        y='kaive',
        color='store_location',
        barmode='group',
        text='kaive',
        title=f'UrbanStyle kuukäive linnade kaupa ({ts})',
        labels={
            'month': 'Kuu',
            'kaive': 'Käive (EUR)',
            'store_location': 'Linn'
        }
    )

    fig.update_traces(
        texttemplate='%{text:.0f}',
        textposition='outside'
    )

    fig.update_layout(
        title_x=0.5,
        plot_bgcolor='white',
        paper_bgcolor='white'
    )

    fig.write_html(f'monthly_city_chart_{ts}.html')

    print("[LOAD] CSV + HTML salvestatud")


# === RUN ===
print("PIPELINE START")

df = extract_orders()

monthly_city = transform_monthly_by_city(df)

load_report(monthly_city)

print("PIPELINE COMPLETE")

print(monthly_city.to_string(index=False))

PIPELINE START
[EXTRACT] Laadin...
[EXTRACT] 20 tellimust laaditud
[TRANSFORM] Arvutan kuuraporti linnade kaupa...
[TRANSFORM] 14 kuu-linna rida
[LOAD] CSV + HTML salvestatud
PIPELINE COMPLETE
  month store_location  tellimusi  kaive  keskmine_tellimus
2024-01        Tallinn          1  89.99              89.99
2024-01          Tartu          1  45.50              45.50
2024-02        Tallinn          2 187.30              93.65
2024-02          Tartu          1  55.00              55.00
2024-03          Parnu          1 210.00             210.00
2024-03        Tallinn          2 178.50              89.25
2024-04          Parnu          1  92.00              92.00
2024-04          Tartu          2 238.00             119.00
2024-05        Tallinn          2 274.00             137.00
2024-05          Tartu          1  88.50              88.50
2024-06          Parnu          2 212.00             106.00
2024-06          Tartu          1 110.00             110.00
2024-07        Tallinn     

RFM segmentide lisamine

In [176]:
import pandas as pd
import plotly.express as px
from datetime import datetime

# === TRANSFORM-RFM ===
def transform_rfm(df, reference_date='2024-08-01'):
    """Arvuta RFM segmendid."""
    print("[TRANSFORM-RFM] Arvutan...")

    df = df.copy()

    df['sale_date'] = pd.to_datetime(df['sale_date'])
    df['total_price'] = pd.to_numeric(df['total_price'])

    ref = pd.to_datetime(reference_date)

    rfm = (
        df.groupby('customer_id')
        .agg(
            last_purchase=('sale_date', 'max'),
            frequency=('sale_date', 'count'),
            monetary=('total_price', 'sum')
        )
        .reset_index()
    )

    rfm['recency_days'] = (
        ref - rfm['last_purchase']
    ).dt.days

    rfm['segment'] = rfm.apply(
        lambda row: 'VIP' if row['monetary'] > 200 and row['frequency'] >= 3
                    else 'Loyal' if row['frequency'] >= 3
                    else 'At Risk' if row['recency_days'] > 120
                    else 'Regular',
        axis=1
    )

    print(f"[TRANSFORM-RFM] {len(rfm)} klienti segmenteeritud")

    return rfm

def load_report(monthly):
    """Salvesta kuuraport CSV ja kuva graafik."""
    print("[LOAD] Salvestan kuuraporti...")

    ts = datetime.now().strftime('%Y%m%d_%H%M')

    monthly.to_csv(
        f'monthly_report_{ts}.csv',
        index=False,
        encoding='utf-8-sig'
    )

    fig = px.bar(
        monthly,
        x='sale_date',
        y='kaive',
        text='kaive',
        title=f'UrbanStyle kuukäive ({ts})',
        labels={
            'sale_date': 'Kuu',
            'kaive': 'Käive (EUR)'
        }
    )

    fig.update_traces(
        texttemplate='%{text:.0f}',
        textposition='outside'
    )

    fig.update_layout(
        title_x=0.5,
        plot_bgcolor='white',
        paper_bgcolor='white'
    )

    fig.show()

    print("[LOAD] Kuuraport salvestatud")

    df = extract_orders()
monthly = transform_monthly(df)
rfm = transform_rfm(df)

load_report(monthly)
load_rfm(rfm)

[TRANSFORM] Arvutan...
[TRANSFORM] 7 kuud
[TRANSFORM-RFM] Arvutan...
[TRANSFORM-RFM] 7 klienti segmenteeritud
[LOAD] Salvestan kuuraporti...


[LOAD] Kuuraport salvestatud
[EXTRACT] Laadin...
[EXTRACT] 20 tellimust laaditud
[LOAD-RFM] Salvestan RFM raporti...


[LOAD-RFM] Fail salvestatud: rfm_segments_20260622_1328.csv
PIPELINE START


TOP VIP klient

In [177]:
top_vip = (
    rfm[rfm['segment'] == 'VIP']
    .sort_values('monetary', ascending=False)
    .head(1)
)

print(top_vip[['customer_id', 'monetary', 'frequency']])

   customer_id  monetary  frequency
0         1001    532.29          4


Kasutame logging moodulit

In [178]:
import pandas as pd
import plotly.express as px
from datetime import datetime

# === TRANSFORM-RFM ===
def transform_rfm(df, reference_date='2024-08-01'):
    """Arvuta RFM segmendid."""
    print("[TRANSFORM-RFM] Arvutan...")

    df = df.copy()

    df['sale_date'] = pd.to_datetime(df['sale_date'])
    df['total_price'] = pd.to_numeric(df['total_price'])

    ref = pd.to_datetime(reference_date)

    rfm = (
        df.groupby('customer_id')
        .agg(
            last_purchase=('sale_date', 'max'),
            frequency=('sale_date', 'count'),
            monetary=('total_price', 'sum')
        )
        .reset_index()
    )

    rfm['recency_days'] = (
        ref - rfm['last_purchase']
    ).dt.days

    rfm['segment'] = rfm.apply(
        lambda row: 'VIP' if row['monetary'] > 200 and row['frequency'] >= 3
                    else 'Loyal' if row['frequency'] >= 3
                    else 'At Risk' if row['recency_days'] > 120
                    else 'Regular',
        axis=1
    )

    print(f"[TRANSFORM-RFM] {len(rfm)} klienti segmenteeritud")

    return rfm

def load_report(monthly):
    """Salvesta kuuraport CSV ja kuva graafik."""
    print("[LOAD] Salvestan kuuraporti...")

    ts = datetime.now().strftime('%Y%m%d_%H%M')

    monthly.to_csv(
        f'monthly_report_{ts}.csv',
        index=False,
        encoding='utf-8-sig'
    )

    fig = px.bar(
        monthly,
        x='sale_date',
        y='kaive',
        text='kaive',
        title=f'UrbanStyle kuukäive ({ts})',
        labels={
            'sale_date': 'Kuu',
            'kaive': 'Käive (EUR)'
        }
    )

    fig.update_traces(
        texttemplate='%{text:.0f}',
        textposition='outside'
    )

    fig.update_layout(
        title_x=0.5,
        plot_bgcolor='white',
        paper_bgcolor='white'
    )

    fig.show()

    print("[LOAD] Kuuraport salvestatud")

    df = extract_orders()
monthly = transform_monthly(df)
rfm = transform_rfm(df)

load_report(monthly)
load_rfm(rfm)

[TRANSFORM] Arvutan...
[TRANSFORM] 7 kuud
[TRANSFORM-RFM] Arvutan...
[TRANSFORM-RFM] 7 klienti segmenteeritud
[LOAD] Salvestan kuuraporti...


[LOAD] Kuuraport salvestatud
[EXTRACT] Laadin...
[EXTRACT] 20 tellimust laaditud
[LOAD-RFM] Salvestan RFM raporti...


[LOAD-RFM] Fail salvestatud: rfm_segments_20260622_1328.csv
PIPELINE START


In [179]:
import pandas as pd
import plotly.express as px
from datetime import datetime

# === TRANSFORM-RFM ===
def transform_rfm(df, reference_date='2024-08-01'):
    """Arvuta RFM segmendid."""
    print("[TRANSFORM-RFM] Arvutan...")

    df = df.copy()

    df['sale_date'] = pd.to_datetime(df['sale_date'])
    df['total_price'] = pd.to_numeric(df['total_price'])

    ref = pd.to_datetime(reference_date)

    rfm = (
        df.groupby('customer_id')
        .agg(
            last_purchase=('sale_date', 'max'),
            frequency=('sale_date', 'count'),
            monetary=('total_price', 'sum')
        )
        .reset_index()
    )

    rfm['recency_days'] = (
        ref - rfm['last_purchase']
    ).dt.days

    rfm['segment'] = rfm.apply(
        lambda row: 'VIP' if row['monetary'] > 200 and row['frequency'] >= 3
                    else 'Loyal' if row['frequency'] >= 3
                    else 'At Risk' if row['recency_days'] > 120
                    else 'Regular',
        axis=1
    )

    print(f"[TRANSFORM-RFM] {len(rfm)} klienti segmenteeritud")

    return rfm

def load_report(monthly):
    """Salvesta kuuraport CSV ja kuva graafik."""
    print("[LOAD] Salvestan kuuraporti...")

    ts = datetime.now().strftime('%Y%m%d_%H%M')

    monthly.to_csv(
        f'monthly_report_{ts}.csv',
        index=False,
        encoding='utf-8-sig'
    )

    fig = px.bar(
        monthly,
        x='sale_date',
        y='kaive',
        text='kaive',
        title=f'UrbanStyle kuukäive ({ts})',
        labels={
            'sale_date': 'Kuu',
            'kaive': 'Käive (EUR)'
        }
    )

    fig.update_traces(
        texttemplate='%{text:.0f}',
        textposition='outside'
    )

    fig.update_layout(
        title_x=0.5,
        plot_bgcolor='white',
        paper_bgcolor='white'
    )

    fig.show()

    print("[LOAD] Kuuraport salvestatud")

    df = extract_orders()
monthly = transform_monthly(df)
rfm = transform_rfm(df)

load_report(monthly)
load_rfm(rfm)

[TRANSFORM] Arvutan...
[TRANSFORM] 7 kuud
[TRANSFORM-RFM] Arvutan...
[TRANSFORM-RFM] 7 klienti segmenteeritud
[LOAD] Salvestan kuuraporti...


[LOAD] Kuuraport salvestatud
[EXTRACT] Laadin...
[EXTRACT] 20 tellimust laaditud
[LOAD-RFM] Salvestan RFM raporti...


[LOAD-RFM] Fail salvestatud: rfm_segments_20260622_1328.csv
PIPELINE START
